# U06_C1 — Multiple Alignment and Conservation
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 6:** Multiple Sequence Alignment
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L16–L17, Practical S8 (Clustal & Jalview)
**Before this:** U05_C3

## 1. What you will learn
- **Explain** what a Multiple Sequence Alignment (MSA) reveals that pairwise alignments cannot (conserved motifs, functional active sites) [LO5].
- **Describe** the progressive alignment strategy (guide tree, profile-profile alignment, "once a gap, always a gap") [LO5].
- **Load** and slice Clustal format alignments using Biopython's `Bio.AlignIO` [LO5].
- **Calculate** per-column conservation and plot a conservation profile along the globin family [LO5].


## 2. Biological question
The human genome encodes multiple distinct globin proteins: hemoglobin alpha (*HBA1*), beta (*HBB*), delta (*HBD*), gamma (*HBG1*), myoglobin (*MB*), neuroglobin (*NGB*), and cytoglobin (*CYGB*). Despite millions of years of evolutionary divergence, all of them bind a haem group and transport oxygen. How do we align all 7 proteins simultaneously to identify the invariant catalytic residues that coordinate the haem iron atom?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U06_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: The Power of Multiple Sequence Alignment
### Theory
Pairwise alignment can only reveal similarity between two sequences. When three or more homologous sequences are aligned:
1. **Conserved columns** pinpoint essential catalytic residues, structural disulfides, or ligand-binding contacts.
2. **Variable columns** indicate neutral, surface-exposed positions tolerant of substitution.
3. **Indel patterns** delineate structural loops connecting conserved secondary structure elements.

Finding the mathematically optimal MSA with dynamic programming across $N$ sequences of length $L$ requires $O(L^N)$ computational time and space, which is **NP-hard** and computationally impossible for more than a few sequences.


## 4. Concept 2: Progressive Alignment Heuristics
### Theory
Algorithms like **ClustalW**, **Clustal Omega**, and **MUSCLE** use the **progressive alignment** heuristic:
1. **All-against-all pairwise alignment**: Calculate distance matrix between all pairs.
2. **Guide tree construction**: Cluster sequences (using UPGMA or Neighbor Joining) into a binary guide tree.
3. **Profile-profile alignment**: Progressively align the most closely related pairs first, then align alignments (profiles) following the guide tree.

**Rule:** *"Once a gap, always a gap."* If an early pairwise step introduces a gap, subsequent steps cannot undo it, potentially propagating early alignment errors.


## 5. Concept 3: Reading and Slicing MSAs with Bio.AlignIO
### Theory
Biopython provides `Bio.AlignIO` to read standard MSA formats (`clustal`, `fasta`, `phylip`):
- `AlignIO.read(path, "clustal")`: Reads a single multiple alignment object (`MultipleSeqAlignment`).
- `len(alignment)`: Number of sequences (rows).
- `alignment.get_alignment_length()`: Number of alignment columns.
- Slicing: `alignment[:, 50]` extracts column 50 across all sequences; `alignment[:, 10:20]` extracts a block of columns.


In [ ]:
# Worked Example: Loading the Human Globin Family MSA
from Bio import AlignIO

aln_file = DATA_DIR / "alignments" / "globin_family.aln"
alignment = AlignIO.read(aln_file, "clustal")

num_seqs = len(alignment)
aln_len = alignment.get_alignment_length()

print(f"Loaded Multiple Alignment with {num_seqs} sequences across {aln_len} columns.\n")
for record in alignment:
    print(f"  {record.id:15} | Length: {len(record.seq)} | Start: {record.seq[:25]}...")


## 6. Concept 4: Column Conservation and Consensus Profile
### Theory
For each alignment column $j$, the fractional identity is the frequency of the most abundant amino acid in that column:
$$\text{Identity}(j) = \frac{\max_{aa} \text{Count}(aa, j)}{\text{Total Sequences}}$$
Columns with $\text{Identity} = 1.0$ (100%) represent invariant residues across all members.


In [ ]:
# Worked Example: Calculating per-column conservation and consensus
import matplotlib.pyplot as plt
from collections import Counter

conservation_scores = []
consensus_seq = []

for col_idx in range(aln_len):
    col_chars = alignment[:, col_idx].upper()
    # Exclude gaps when determining dominant residue
    valid_chars = [c for c in col_chars if c != "-"]
    if valid_chars:
        counts = Counter(valid_chars)
        most_common_aa, count = counts.most_common(1)[0]
        score = count / num_seqs
        consensus_seq.append(most_common_aa)
    else:
        score = 0.0
        consensus_seq.append("-")
    conservation_scores.append(score)

print(f"Consensus sequence length: {len(consensus_seq)}")
print(f"First 40 residues of consensus: {''.join(consensus_seq[:40])}")

# Plot conservation profile
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(conservation_scores, color="darkblue", lw=1.2)
ax.axhline(1.0, color="red", linestyle="--", alpha=0.7, label="100% Invariant")
ax.set_xlabel("Alignment Column Index")
ax.set_ylabel("Fractional Conservation")
ax.set_title("Conservation Profile across 7 Human Globins")
ax.set_ylim(-0.05, 1.1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUT_DIR / "globin_conservation_profile.png", dpi=100)
plt.show()

# Find conserved positions
conserved_cols = [i for i, s in enumerate(conservation_scores) if s >= 0.7]
print(f"\nFound {len(conserved_cols)} conserved columns (>=70% identity) across all 7 globins:")
for c in conserved_cols:
    print(f"  Column {c:3}: Dominant {consensus_seq[c]} (Score: {conservation_scores[c]:.2f})")


## 7. Compare with the Web Tool
In Practical S8 (Jalview / Clustal Omega):
- Invariant columns are marked with an asterisk (`*`) beneath the alignment.
- Conserved substitutions within strongly similar properties are marked with a colon (`:`).
- Weakly similar substitutions are marked with a period (`.`).
Our Python calculation identifies the same invariant columns, notably the proximal Histidine (coordinating the haem iron) and distal Histidine.


## 8. Summary
- Progressive alignment builds an MSA step-by-step from pairwise distances and a guide tree.
- `Bio.AlignIO.read()` parses alignments into `MultipleSeqAlignment` objects.
- Slicing `alignment[:, col]` accesses column slices across all sequences.
- Invariant columns identify core functional residues essential for protein mechanism.


## 9. Next Steps
- Analyze globin conservation in `U06_P_Multiple_Alignment.ipynb`.
- Self-test multiple sequence alignment concepts in `U06_Q_Multiple_Alignment.ipynb`.
